# PyTorch Fundamentals

## Manipulating Tensors

### _Setting Default Device to CUDA_

In [1]:
import torch

print(torch.get_default_device())

device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else 'cpu'

torch.set_default_device(device=device)

print(torch.get_default_device())

SEED = 69
torch.manual_seed(seed=SEED)

cpu
cuda:0


### _Addition_

In [2]:
tensor = torch.Tensor([1, 2, 3])
tensor + 10

tensor([11., 12., 13.])

### _Multiplication_

In [3]:
tensor * 10

tensor([10., 20., 30.])

### _Substraction_

In [4]:
tensor - 10

tensor([-9., -8., -7.])

### _Division_

In [5]:
tensor / 10

tensor([0.1000, 0.2000, 0.3000])

### _Checking Tensor DataType_

In [6]:
(tensor / 10).dtype

torch.float32

### _Element Wise Multiplication_

In [7]:
tensor * tensor

tensor([1., 4., 9.])

### _Matrix Muliplication_

In [8]:
torch.matmul(tensor, tensor)

tensor(14.)

We can do matrix mulitiplication manually but it's not recommended. The built-in `torch.matmul` method is faster.

In [9]:
%%time

# manual matrix multiplication

value = 0

for i in range(len(tensor)):
    value += tensor[i] + tensor[i]

value

CPU times: user 247 μs, sys: 15 μs, total: 262 μs
Wall time: 1.8 ms


tensor(12.)

In [10]:
%%time

def manual_matrix_multiply(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """Multiply a (m, n) by b (n, p) -> (m, p) with explicit loops.
    1-D inputs are handled the way torch.matmul handles them."""
    a_shape, b_shape = tuple(a.shape), tuple(b.shape)

    # Treat vectors as matrices: (n,) -> (1, n) on the left, (n,) -> (n, 1) on the right
    a_was_1d, b_was_1d = a.dim() == 1, b.dim() == 1
    if a_was_1d:
        a = a.unsqueeze(0)
    if b_was_1d:
        b = b.unsqueeze(1)

    if a.dim() != 2 or b.dim() != 2:
        raise ValueError(
            f"expected 1-D or 2-D tensors, got shapes {a_shape} and {b_shape}"
        )

    m, n = a.shape
    n2, p = b.shape
    if n != n2:
        raise ValueError(
            f"cannot multiply {a_shape} by {b_shape}: "
            f"inner dimensions {n} and {n2} must match"
        )

    out = torch.zeros(m, p, dtype=torch.result_type(a, b), device=a.device)
    for i in range(m):  # each row of a
        for j in range(p):  # each column of b
            total = 0
            for k in range(n):  # walk along the shared dimension
                total += a[i, k] * b[k, j]
            out[i, j] = total

    # Undo the vector-to-matrix conversion, as torch.matmul does
    if a_was_1d:
        out = out.squeeze(0)
    if b_was_1d:
        out = out.squeeze(-1)
    return out


t1 = torch.Tensor([[1, 2, 3], [4, 5, 6]])
t2 = torch.Tensor([[6, 5], [4, 3], [2, 1]])

print(f'manual multiplication: {manual_matrix_multiply(t1, t2)}')
print(f'operator multiplication: {t1 @ t2}')

manual multiplication: tensor([[20., 14.],
        [56., 41.]])
operator multiplication: tensor([[20., 14.],
        [56., 41.]])
CPU times: user 446 μs, sys: 929 μs, total: 1.38 ms
Wall time: 2.63 ms


In [11]:
tensor_A = torch.tensor([[1, 2],
                         [3, 4],
                         [5, 6]], dtype=torch.float32)

tensor_B = torch.tensor([[7, 10],
                         [8, 11], 
                         [9, 12]], dtype=torch.float32)

```python
torch.matmul(tensor_A, tensor_B) 
```

will produce 

```python
RuntimeError: mat1 and mat2 shapes cannot be multiplied (3x2 and 3x2)
```

Because much of deep learning is multiplying and performing operations on matrices and matrices have a strict rule about what shapes and sizes can be combined, one of the most common errors you'll run into in deep learning is shape mismatches.


We can make matrix multiplication work between tensor_A and tensor_B by making their inner dimensions match.

One of the ways to do this is with a transpose (switch the dimensions of a given tensor).

We can perform transposes in PyTorch using either:

In [12]:
print('View of tensor_A and tensor_B')
print(tensor_A)
print(tensor_B)

View of tensor_A and tensor_B
tensor([[1., 2.],
        [3., 4.],
        [5., 6.]], device='cuda:0')
tensor([[ 7., 10.],
        [ 8., 11.],
        [ 9., 12.]], device='cuda:0')


In [13]:
print('View of tensor_A and transpose of tensor_B')
print(tensor_A)
print(tensor_B.T)

View of tensor_A and transpose of tensor_B
tensor([[1., 2.],
        [3., 4.],
        [5., 6.]], device='cuda:0')
tensor([[ 7.,  8.,  9.],
        [10., 11., 12.]], device='cuda:0')


In [14]:
# The operation works when tensor_B is transposed
print(f"Original shapes: tensor_A = {tensor_A.shape}, tensor_B = {tensor_B.shape}\n")
print(f"New shapes: tensor_A = {tensor_A.shape} (same as above), tensor_B.T = {tensor_B.T.shape}\n")
print(f"Multiplying: {tensor_A.shape} * {tensor_B.T.shape} <- inner dimensions match\n")
print("Output:\n")
output = torch.matmul(tensor_A, tensor_B.T)
print(output) 
print(f"\nOutput shape: {output.shape}")

Original shapes: tensor_A = torch.Size([3, 2]), tensor_B = torch.Size([3, 2])

New shapes: tensor_A = torch.Size([3, 2]) (same as above), tensor_B.T = torch.Size([2, 3])

Multiplying: torch.Size([3, 2]) * torch.Size([2, 3]) <- inner dimensions match

Output:

tensor([[ 27.,  30.,  33.],
        [ 61.,  68.,  75.],
        [ 95., 106., 117.]], device='cuda:0')

Output shape: torch.Size([3, 3])


Neural networks are full of matrix multiplications and dot products.

The `torch.nn.Linear()` module (we'll see this in action later on), also known as a feed-forward layer or fully connected layer, implements a matrix multiplication between an input `x` and a weights matrix `A`.

$$ y = x\cdot{A^T} + b $$

Where:

$x$ is the input to the layer (deep learning is a stack of layers like `torch.nn.Linear()` and others on top of each other).
$A$ is the weights matrix created by the layer, this starts out as random numbers that get adjusted as a neural network learns to better represent patterns in the data (notice the `T`, that's because the weights matrix gets transposed).
Note: You might also often see `W` or another letter like `X` used to showcase the weights matrix.
`b` is the bias term used to slightly offset the weights and inputs.
`y` is the output (a manipulation of the input in the hopes to discover patterns in it).
This is a linear function (you may have seen something like $y = mx+b$ in high school or elsewhere), and can be used to draw a straight line!

Let's play around with a linear layer.

Try changing the values of in_features and o

In [15]:
linear = torch.nn.Linear(in_features=2, out_features=10)

tensor_A = torch.tensor([[1, 2],
                         [3, 4],
                         [5, 6]], dtype=torch.float32)
x = tensor_A
output = linear(x)

print(f'input shape: {x.shape}\n')
print(f'output:\n{output}\n\nOutput shape: {output.shape}')

input shape: torch.Size([3, 2])

output:
tensor([[-0.8545,  0.9589,  0.4001,  1.8893,  0.5006, -0.5367, -0.8364, -1.4788,
          0.6377, -0.3705],
        [-2.5635,  1.4136,  0.0657,  3.4187,  1.2260, -0.0932, -2.1764, -3.1181,
          0.2882, -0.3073],
        [-4.2724,  1.8683, -0.2688,  4.9480,  1.9515,  0.3504, -3.5165, -4.7574,
         -0.0613, -0.2441]], device='cuda:0', grad_fn=<AddmmBackward0>)

Output shape: torch.Size([3, 10])


### _Positional min/max_

We can also find the index of a tensor where the max or minimum occurs with `torch.argmax()` and `torch.argmin()` respectively.

This is helpful incase you just want the position where the highest (or lowest) value is and not the actual value itself (we'll see this in a later section when using the softmax activation function).

In [16]:
tensor = torch.arange(0, 100, 10)
print(f'Tensor: {tensor}')

print(f'index where max value occurs: {tensor.argmax()}')
print(f'index where min value occurs: {tensor.argmin()}')

Tensor: tensor([ 0, 10, 20, 30, 40, 50, 60, 70, 80, 90], device='cuda:0')
index where max value occurs: 9
index where min value occurs: 0


### _Change tensor datatype_

As mentioned, a common issue with deep learning operations is having your tensors in different datatypes.

If one tensor is in `torch.float64` and another is in `torch.float32`, you might run into some errors.

But there's a fix.

We can change the datatypes of tensors using `torch.Tensor.type(dtype=None)` where the dtype parameter is the datatype you'd like to use.

First we'll create a tensor and check its datatype (the default is `torch.float32`).

In [17]:
tensor_float32 = torch.arange(1., 100., 1.)
print(tensor_float32)
print(f'\ndatatype: {tensor_float32.dtype}')

tensor([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13., 14.,
        15., 16., 17., 18., 19., 20., 21., 22., 23., 24., 25., 26., 27., 28.,
        29., 30., 31., 32., 33., 34., 35., 36., 37., 38., 39., 40., 41., 42.,
        43., 44., 45., 46., 47., 48., 49., 50., 51., 52., 53., 54., 55., 56.,
        57., 58., 59., 60., 61., 62., 63., 64., 65., 66., 67., 68., 69., 70.,
        71., 72., 73., 74., 75., 76., 77., 78., 79., 80., 81., 82., 83., 84.,
        85., 86., 87., 88., 89., 90., 91., 92., 93., 94., 95., 96., 97., 98.,
        99.], device='cuda:0')

datatype: torch.float32


In [18]:
tensor_float16 = tensor_float32.type(torch.float16)
print(tensor_float16)
print(f'\ndatatype: {tensor_float16.dtype}')

tensor([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13., 14.,
        15., 16., 17., 18., 19., 20., 21., 22., 23., 24., 25., 26., 27., 28.,
        29., 30., 31., 32., 33., 34., 35., 36., 37., 38., 39., 40., 41., 42.,
        43., 44., 45., 46., 47., 48., 49., 50., 51., 52., 53., 54., 55., 56.,
        57., 58., 59., 60., 61., 62., 63., 64., 65., 66., 67., 68., 69., 70.,
        71., 72., 73., 74., 75., 76., 77., 78., 79., 80., 81., 82., 83., 84.,
        85., 86., 87., 88., 89., 90., 91., 92., 93., 94., 95., 96., 97., 98.,
        99.], device='cuda:0', dtype=torch.float16)

datatype: torch.float16


In [19]:
tensor_int8 = tensor_float32.type(torch.int8)
print(tensor_int8)
print(f'\ndataype: {tensor_int8.dtype}')

tensor([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18,
        19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36,
        37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54,
        55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72,
        73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90,
        91, 92, 93, 94, 95, 96, 97, 98, 99], device='cuda:0', dtype=torch.int8)

dataype: torch.int8


### _Reshaping, stacking and squeezing and unsqueezing_

Often times you'll want to reshape or change the dimensions of your tensors without actually changing the values inside them.

To do so, some popular methods are:

Reshaping, stacking, squeezing and unsqueezing¶
Often times you'll want to reshape or change the dimensions of your tensors without actually changing the values inside them.

To do so, some popular methods are:

| Method | One-line description |
|---|---|
| `torch.reshape(input, shape)` | Reshapes `input` to `shape` (if compatible), can also use `torch.Tensor.reshape()`. |
| `Tensor.view(shape)` | Returns a view of the original tensor in a different `shape` but shares the same data as the original tensor. |
| `torch.stack(tensors, dim=0)` | Concatenates a sequence of `tensors` along a new dimension (`dim`), all `tensors` must be same size. |
| `torch.squeeze(input)` | Squeezes `input` to remove all the dimensions with value `1`. |
| `torch.unsqueeze(input, dim)` | Returns `input` with a dimension value of `1` added at `dim`. |
| `torch.permute(input, dims)` | Returns a view of the original `input` with its dimensions permuted (rearranged) to `dims`. |

\
**_Why do any of these?_**

Because deep learning models (neural networks) are all about manipulating tensors in some way. And because of the rules of matrix multiplication, if you've got shape mismatches, you'll run into errors. These methods help you make sure the right elements of your tensors are mixing with the right elements of other tensors.


In [20]:
import torch

x = torch.arange(1., 8.)
x, x.shape

(tensor([1., 2., 3., 4., 5., 6., 7.], device='cuda:0'), torch.Size([7]))

In [21]:
# add an extra dimension
x_reshaped = x.reshape(1,7)
x_reshaped, x_reshaped.shape

(tensor([[1., 2., 3., 4., 5., 6., 7.]], device='cuda:0'), torch.Size([1, 7]))

We can also change the view with `torch.view()`

In [22]:
z = x.view(1, 7)
z, z.shape

(tensor([[1., 2., 3., 4., 5., 6., 7.]], device='cuda:0'), torch.Size([1, 7]))

In [26]:
z_alter = x.view(7, 1)
z_alter, z_alter.shape

(tensor([[5.],
         [2.],
         [3.],
         [4.],
         [5.],
         [6.],
         [7.]], device='cuda:0'),
 torch.Size([7, 1]))

Remember though, changing the view of a tensor with `torch.view()` really only creates a new view of the same tensor.

So changing the view changes the original tensor too.

In [ ]:
z[:, 0] = 5 
z, z_alter, x

(tensor([[5., 2., 3., 4., 5., 6., 7.]], device='cuda:0'),
 tensor([5., 2., 3., 4., 5., 6., 7.], device='cuda:0'))

if we wanted to stack our new tensor on top of itself five times, we could do so with `torch.stack()`

In [24]:
x_stacked_0 = torch.stack([x,x,x,x], dim=0)
x_stacked_0

tensor([[5., 2., 3., 4., 5., 6., 7.],
        [5., 2., 3., 4., 5., 6., 7.],
        [5., 2., 3., 4., 5., 6., 7.],
        [5., 2., 3., 4., 5., 6., 7.]], device='cuda:0')

In [25]:
x_stacked_1 = torch.stack([x,x,x,x], dim=1)
x_stacked_1

tensor([[5., 5., 5., 5.],
        [2., 2., 2., 2.],
        [3., 3., 3., 3.],
        [4., 4., 4., 4.],
        [5., 5., 5., 5.],
        [6., 6., 6., 6.],
        [7., 7., 7., 7.]], device='cuda:0')

How about removing all single dimensions from a tensor?

To do so you can use `torch.squeeze()` 

In [27]:
print(f"Previous tensor: {x_reshaped}")
print(f"Previous shape: {x_reshaped.shape}")

# Remove extra dimension from x_reshaped
x_squeezed = x_reshaped.squeeze()
print(f"\nNew tensor: {x_squeezed}")
print(f"New shape: {x_squeezed.shape}")

Previous tensor: tensor([[5., 2., 3., 4., 5., 6., 7.]], device='cuda:0')
Previous shape: torch.Size([1, 7])

New tensor: tensor([5., 2., 3., 4., 5., 6., 7.], device='cuda:0')
New shape: torch.Size([7])


### _Self Practice Reshaping and Squeezing_

In [31]:
low, high = 1, 10
y = low + (high - low) * torch.rand(10)
y, y.min(), y.max(), y.argmin(), y.argmax()

(tensor([5.3662, 2.6781, 4.2953, 4.6642, 8.6658, 6.2778, 2.2476, 5.8240, 1.1299,
         2.9281], device='cuda:0'),
 tensor(1.1299, device='cuda:0'),
 tensor(8.6658, device='cuda:0'),
 tensor(8, device='cuda:0'),
 tensor(4, device='cuda:0'))

In [37]:
y_row = y.view(10, 1)
y_col = y.view(1, 10)
y_col, y_col.shape, y_row, y_row.shape

(tensor([[5.3662, 2.6781, 4.2953, 4.6642, 8.6658, 6.2778, 2.2476, 5.8240, 1.1299,
          2.9281]], device='cuda:0'),
 torch.Size([1, 10]),
 tensor([[5.3662],
         [2.6781],
         [4.2953],
         [4.6642],
         [8.6658],
         [6.2778],
         [2.2476],
         [5.8240],
         [1.1299],
         [2.9281]], device='cuda:0'),
 torch.Size([10, 1]))

In [43]:
y_col = y_col.squeeze()
y_row = y_row.squeeze()
y_col, y_col.shape, y_row, y_row.shape

(tensor([5.3662, 2.6781, 4.2953, 4.6642, 8.6658, 6.2778, 2.2476, 5.8240, 1.1299,
         2.9281], device='cuda:0'),
 torch.Size([10]),
 tensor([5.3662, 2.6781, 4.2953, 4.6642, 8.6658, 6.2778, 2.2476, 5.8240, 1.1299,
         2.9281], device='cuda:0'),
 torch.Size([10]))

In [ ]:
y_col_stack = torch.stack([y_col, y_col, y_col, y_col, y_col, y_col, y_col, y_col, y_col, y_col], dim=1)
y_col_stack, y_col_stack.shape

(tensor([[[5.3662, 2.6781, 4.2953, 4.6642, 8.6658, 6.2778, 2.2476, 5.8240,
           1.1299, 2.9281],
          [5.3662, 2.6781, 4.2953, 4.6642, 8.6658, 6.2778, 2.2476, 5.8240,
           1.1299, 2.9281],
          [5.3662, 2.6781, 4.2953, 4.6642, 8.6658, 6.2778, 2.2476, 5.8240,
           1.1299, 2.9281],
          [5.3662, 2.6781, 4.2953, 4.6642, 8.6658, 6.2778, 2.2476, 5.8240,
           1.1299, 2.9281],
          [5.3662, 2.6781, 4.2953, 4.6642, 8.6658, 6.2778, 2.2476, 5.8240,
           1.1299, 2.9281],
          [5.3662, 2.6781, 4.2953, 4.6642, 8.6658, 6.2778, 2.2476, 5.8240,
           1.1299, 2.9281],
          [5.3662, 2.6781, 4.2953, 4.6642, 8.6658, 6.2778, 2.2476, 5.8240,
           1.1299, 2.9281],
          [5.3662, 2.6781, 4.2953, 4.6642, 8.6658, 6.2778, 2.2476, 5.8240,
           1.1299, 2.9281],
          [5.3662, 2.6781, 4.2953, 4.6642, 8.6658, 6.2778, 2.2476, 5.8240,
           1.1299, 2.9281],
          [5.3662, 2.6781, 4.2953, 4.6642, 8.6658, 6.2778, 2.2476, 5.8240

### Pytorch Tensors and Numpy

Since NumPy is a popular Python numerical computing library, PyTorch has functionality to interact with it nicely.

The two main methods you'll want to use for NumPy to PyTorch (and back again) are:

-`torch.from_numpy(ndarray)` - NumPy array -> PyTorch tensor.
-`torch.Tensor.numpy()` - PyTorch tensor -> NumPy array.

> Note: By default, NumPy arrays are created with the datatype float64 and if you convert it to a PyTorch tensor, it'll keep the same datatype (as above).
>
> However, many PyTorch calculations default to using float32.
>
> So if you want to convert your NumPy array (`float64`) -> PyTorch tensor (`float64`) -> PyTorch tensor (`float32`), you can use tensor = `torch.from_numpy(array).type(torch.float32)`.

In [45]:
import torch
import numpy as np

array = np.arange(1.0, 8.0)
tensor = torch.from_numpy(array).type(torch.float32)
array, tensor

(array([1., 2., 3., 4., 5., 6., 7.]), tensor([1., 2., 3., 4., 5., 6., 7.]))

In [46]:
# change the array keep the tensor
array = array + 1
array, tensor

(array([2., 3., 4., 5., 6., 7., 8.]), tensor([1., 2., 3., 4., 5., 6., 7.]))

If we want to go from PyTorch tensor to NumPy array, we can call `tensor.numpy()`

In [49]:
tensor = torch.ones(7).cpu()
numpy_tensor = tensor.numpy()
tensor, numpy_tensor

(tensor([1., 1., 1., 1., 1., 1., 1.]),
 array([1., 1., 1., 1., 1., 1., 1.], dtype=float32))

### Reproduciblity (trying to make the random out of random)

As we learn more about neural networks and machine learning, we started to discover how much randomness plays a part. Well, pseudorandomness that is. Because after all, as they're designed, a computer is fundamentally deterministic, each step is predictable, so the randomness they create are simulated randomness, though there is debate on this too, but since we're not computer scientists, let's find out more ourselves. How does this relate to neual networks and deep learning then? We've discussed neural networks start with random numbers to describe patterns in data and try to imporove those random numbers using tnesor opoerations to better describe parrterns in data. Although randomness is nice and powerful, sometimes we'd like there to be a little less randomness.


For example, we created an algorithm capable of achieving X performance and then a friend tries it out  to verify that we're not crazy. How could they do such a thing? In other words, can our friend get the same reesults on their computer running the same code as we get on ours? Let's see a brief example of reproducibility in PyTorch.

In [50]:
random_tensor_A = torch.rand(3, 4)
random_tensor_B = torch.rand(3, 4)

print(f"Tensor A:\n{random_tensor_A}\n")
print(f"Tensor B:\n{random_tensor_B}\n")
print(f"Does Tensor A equal Tensor B? (anywhere)")
random_tensor_A == random_tensor_B

Tensor A:
tensor([[0.1514, 0.6720, 0.6474, 0.8058],
        [0.7808, 0.1342, 0.1794, 0.0273],
        [0.0663, 0.8314, 0.0978, 0.1574]], device='cuda:0')

Tensor B:
tensor([[0.1482, 0.2569, 0.7138, 0.4434],
        [0.9795, 0.9382, 0.1500, 0.7639],
        [0.0197, 0.0169, 0.3480, 0.8557]], device='cuda:0')

Does Tensor A equal Tensor B? (anywhere)


tensor([[False, False, False, False],
        [False, False, False, False],
        [False, False, False, False]], device='cuda:0')

Jus as we might've expected, the tensors come out with different values. However, what if we wanted to create two random tensors with the same values. As in, the tensors would still contain random values but they would be of the same flavor. This is where `torch.manual_seed(seed)` comes in. where `seed` is an integer that flavours the randomness. Let's try it out by creating some more flavoured randomed tensors.

In [52]:
SEED = 69

# set the random seed
torch.manual_seed(seed=SEED)
random_tensor_C = torch.rand(3, 4)

# have to reset teh seed every time a new rand() is called
# without this, tensor_D would be different to tensor_C

torch.random.manual_seed(seed=SEED)
random_tensor_D = torch.rand(3, 4)

print(f"Tensor C:\n{random_tensor_C}\n")
print(f"Tensor D:\n{random_tensor_D}\n")
print(f"Does Tensor C equal Tensor D? (anywhere)")
random_tensor_C == random_tensor_D

Tensor C:
tensor([[0.2492, 0.1466, 0.6058, 0.5550],
        [0.3204, 0.5613, 0.6839, 0.8568],
        [0.5141, 0.7424, 0.9076, 0.2492]], device='cuda:0')

Tensor D:
tensor([[0.2492, 0.1466, 0.6058, 0.5550],
        [0.3204, 0.5613, 0.6839, 0.8568],
        [0.5141, 0.7424, 0.9076, 0.2492]], device='cuda:0')

Does Tensor C equal Tensor D? (anywhere)


tensor([[True, True, True, True],
        [True, True, True, True],
        [True, True, True, True]], device='cuda:0')

Nice!
It looks like setting the seed worked.

> Resource: What we've just covered only scratches the surface of reproducibility in PyTorch. For more, on reproducibility in general and random seeds, let's checkout:
> 
> - The PyTorch reproducibility documentation (a good exercise would be to read through this for 10-minutes and even if you don't understand it now, being aware of it is important).
> - The Wikipedia random seed page (this'll give a good overview of random seeds and pseudorandomness in general).